## Usando pandas aplicado ao banco Northwind

#### Blocos de preparação

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
import northwind as nw

pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
produtos     = nw.tabela("products")
categorias   = nw.tabela("categories")
clientes     = nw.tabela("customers")
pedidos      = nw.tabela("orders")
itens        = nw.tabela("order_details")
funcionarios = nw.tabela("employees")
fornecedores = nw.tabela("suppliers")

produtos.head()

,product_id,product_name,supplier_id,category_id,quantity_per_unit,unit_price,units_in_stock,units_on_order,reorder_level,discontinued
0,1,Chai,8,1,10 boxes x 30 bags,18.00,39,0,10,1
1,2,Chang,1,1,24 - 12 oz bottles,19.00,17,40,25,1
2,3,Aniseed Syrup,1,2,12 - 550 ml bottles,10.00,13,70,25,0
3,4,Chef Anton's Cajun Seasoning,2,2,48 - 6 oz jars,22.00,53,0,0,0
4,5,Chef Anton's Gumbo Mix,2,2,36 boxes,21.35,0,0,0,1


`info()` é o `\d tabela` do pandas: mostra colunas, tipos e quantos valores não nulos existem.

In [11]:
produtos.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product_id         77 non-null     int64  
 1   product_name       77 non-null     str    
 2   supplier_id        77 non-null     int64  
 3   category_id        77 non-null     int64  
 4   quantity_per_unit  77 non-null     str    
 5   unit_price         77 non-null     float64
 6   units_in_stock     77 non-null     int64  
 7   units_on_order     77 non-null     int64  
 8   reorder_level      77 non-null     int64  
 9   discontinued       77 non-null     int64  
dtypes: float64(1), int64(7), str(2)
memory usage: 6.1 KB


---

## Dicionário de tradução



| SQL | Pandas |
|---|---|
| `SELECT col1, col2` | `df[["col1", "col2"]]` |
| `WHERE cond` | `df[df["col"] > x]` ou `df.query("col > x")` |
| `WHERE col BETWEEN a AND b` | `df["col"].between(a, b)` |
| `WHERE col LIKE '%x%'` | `df["col"].str.contains("x")` |
| `WHERE col IN (...)` | `df["col"].isin([...])` |
| `WHERE col IS NULL` | `df["col"].isna()` |
| `ORDER BY col DESC` | `df.sort_values("col", ascending = False)` |
| `ORDER BY col DESC LIMIT n` | `df.nlargest(n, "col")` |
| `DISTINCT` | `df.drop_duplicates()` |
| `INNER JOIN` | `df.merge(outro, on = "chave")` |
| `LEFT JOIN` | `df.merge(outro, on = "chave", how = "left")` |
| `FULL OUTER JOIN` | `df.merge(outro, on = "chave", how = "outer")` |
| `CROSS JOIN` | `df.merge(outro, how = "cross")` |
| `GROUP BY ... COUNT/SUM/AVG` | `df.groupby("col").agg(...)` |
| `COUNT(DISTINCT col)` | `.nunique()` |
| `HAVING cond` | filtrar **depois** do `groupby` |
| `UNION ALL` | `pd.concat([df1, df2])` |
| `UNION` | `pd.concat([df1, df2]).drop_duplicates()` |
| `WITH nome AS (...)` | uma variável: `nome = ...` |
| `AVG(x) OVER (PARTITION BY g)` | `df.groupby("g")["x"].transform("mean")` |
| `RANK() OVER (PARTITION BY g ORDER BY x)` | `df.groupby("g")["x"].rank(method = "min")` |
| `LAG(x)` / `LEAD(x)` | `df["x"].shift(1)` / `df["x"].shift(-1)` |
| `DATE_TRUNC('month', d)` | `df["d"].dt.to_period("M")` |
| `EXTRACT(YEAR FROM d)` | `df["d"].dt.year` |
| `COALESCE(a, b)` | `df["a"].fillna(df["b"])` |
| `CASE WHEN` | `np.select([cond1, cond2], [v1, v2], default = v3)` |
| `CAST(x AS NUMERIC)` | `df["x"].astype("float64")` |
| `NOT EXISTS` | `merge(..., how = "left", indicator = True)` + `_merge == "left_only"` |


**OBS.:** uma CTE não é uma construção especial, mas é uma variável. Quando você escreve `WITH vendas AS (...)`, em pandas você escreve `vendas = ...`. Toda a fluência em encadear CTE's já é fluência em encadear DataFrames.

---

### Parte 1 - Filtro, ordenação e projeção

#### Exemplo 1 - `WHERE` e `ORDER BY`

Exercício 05:

```sql
  SELECT p.product_id, 
         p.product_name
    FROM products p
   WHERE p.unit_price > 50
ORDER BY p.unit_price DESC
```

In [ ]:
sql = '''
  SELECT p.product_id, 
         p.product_name
    FROM products p
   WHERE p.unit_price > 50
ORDER BY p.unit_price DESC
'''

caros = (produtos.loc[produtos["unit_price"] > 50, ["product_id", "product_name", "unit_price"]].sort_values("unit_price", ascending = False))

nw.conferir(sql, caros)
caros

`.loc[linhas, colunas]` faz o `WHERE` e o `SELECT` de uma vez. É o formato mais próximo do SQL.

#### Exemplo 2 - `BETWEEN`, `LIKE` e funções de texto

Exercícios 06, 09, 10 e 11 da mesma Lista 01:

In [ ]:
entre_50_e_200 = produtos[produtos["unit_price"].between(50, 200)].sort_values("product_name")
contem_tofu    = produtos[produtos["product_name"].str.contains("Tofu")]
comeca_com_t   = produtos[produtos["product_name"].str.startswith("T")]
nome_com_4     = produtos[produtos["product_name"].str.len() == 4]

for nome, df in [("BETWEEN 50 AND 200", entre_50_e_200),
                 ("LIKE '%Tofu%'", contem_tofu),
                 ("LIKE 'T%'", comeca_com_t),
                 ("LENGTH(...) = 4", nome_com_4)]:
    print(f"{nome:<22} -> {len(df):>3} linhas")

O acessador `.str` expõe os métodos de texto do Python vetorizados: `contains`, `startswith`, `len`, `upper`, `replace`. É onde vive todo o `LIKE` e as funções de string do SQL.

#### Exemplo 3 - `ORDER BY ... LIMIT`

Lista básica 02, exercícios 38 e 39:

```sql
  SELECT product_name, 
         CAST(unit_price AS NUMERIC(10,2)) AS UnitPrice
    FROM products
ORDER BY unit_price DESC
   LIMIT 2
```

In [ ]:
produtos.nlargest(2, "unit_price")[["product_name", "unit_price"]]

`nlargest(n, col)` é mais direto e mais rápido que `sort_values().head(n)`, porque não precisa ordenar a tabela inteira. Existe também `nsmallest`.

---

### Parte 2 - Join's

#### Exemplo 4 - `INNER JOIN`

Lista básica 01, exercício 07:

```sql
SELECT p.product_id, 
       p.product_name, 
       c.category_id
  FROM products   p
  JOIN categories c ON c.category_id = p.category_id
 WHERE c.category_id IN (2, 4, 6)
```

In [ ]:
sql = '''
SELECT p.product_id, 
       p.product_name, 
       c.category_id
  FROM products   p
  JOIN categories c ON c.category_id = p.category_id
 WHERE c.category_id IN (2, 4, 6)
'''

selecionados = (produtos.merge(categorias, on = "category_id").query("category_id in [2, 4, 6]")[["product_id", "product_name", "category_id"]])

nw.conferir(sql, selecionados)
selecionados.head()

`merge` usa `how = "inner"` por padrão, igual ao `JOIN` sem qualificador do SQL. Quando as colunas de junção têm nomes diferentes, use `left_on =` e `right_on =`.

#### Exemplo 5 - `LEFT JOIN`

Lista básica 02, exercício 36 — funcionário e seu chefe:

```sql
   SELECT e1.first_name, 
          e1.last_name, 
          e2.first_name, 
          e2.last_name
     FROM employees e1
LEFT JOIN employees e2 ON e2.employee_id = e1.reports_to
```

In [ ]:
chefes = funcionarios[["employee_id", "first_name", "last_name"]]

hierarquia = (funcionarios.merge(chefes, left_on = "reports_to", right_on = "employee_id", how = "left", suffixes = ("", "_chefe"))
              [["first_name", "last_name", "first_name_chefe", "last_name_chefe"]])

hierarquia

`suffixes=("", "_chefe")` resolve a colisão de nomes que no SQL você resolveria com os apelidos `e1` e `e2`. Deixar o primeiro sufixo vazio mantém as colunas da esquerda com o nome original.

### Exemplo 6 - Join com `DISTINCT`

Lista básica 02, exercício 35: clientes e os produtos que compraram

```sql
  SELECT DISTINCT 
         c.company_name, 
         p.product_name
    FROM customers     c
    JOIN orders        o  ON o.customer_id = c.customer_id
    JOIN order_details od ON od.order_id = o.order_id
    JOIN products      p  ON p.product_id = od.product_id
ORDER BY c.company_name, p.product_name
```

In [ ]:
sql = '''
  SELECT DISTINCT 
         c.company_name, 
         p.product_name
    FROM customers     c
    JOIN orders        o  ON o.customer_id = c.customer_id
    JOIN order_details od ON od.order_id = o.order_id
    JOIN products      p  ON p.product_id = od.product_id
ORDER BY c.company_name, p.product_name
'''

cliente_produto = (clientes.merge(pedidos, on = "customer_id").merge(itens, on = "order_id").merge(produtos, on = "product_id")
                   [["company_name", "product_name"]].drop_duplicates().sort_values(["company_name", "product_name"]))

nw.conferir(sql, cliente_produto)
cliente_produto.head()

O encadeamento de `merge` lê na mesma ordem do `FROM ... JOIN ... JOIN`. A diferença é que aqui a ordem importa para o desempenho e quem decide é o usuário, não há otimizador reordenando.

---

### Parte 3 - Agregação

#### Exemplo 7 - `GROUP BY` com expressão calculada

Total de vendas por categoria. Note o padrão do Northwind: o valor da linha é `quantity * unit_price * (1 - discount)`. Vale calcular essa coluna uma vez e reaproveitar.

In [ ]:
itens_valorizados = itens.assign(valor = itens["quantity"] * itens["unit_price"] * (1 - itens["discount"])
)

vendas_categoria = (itens_valorizados.merge(produtos[["product_id", "category_id"]], on = "product_id")
                    .merge(categorias[["category_id", "category_name"]], on = "category_id")
                    .groupby(["category_id", "category_name"], as_index = False).agg(total_vendas = ("valor", "sum"))
                    .sort_values("total_vendas", ascending = False))

vendas_categoria

Dois detalhes que economizam muito tempo depois:

- **`as_index=False`**: mantém as colunas do agrupamento como colunas normais, em vez de virarem índice. É o comportamento que corresponde ao `GROUP BY` do SQL.
- **Agregação nomeada**: `agg(nome=("coluna", "função"))` é o equivalente direto de `SUM(valor) AS total_vendas`.

### Exemplo 8 - CTE com `HAVING` (Padrão muito importante)

Lista avançada 05, exercício 01. Aqui está a tradução conceitual central da fase:

```sql
WITH vendas_cliente AS
(
      SELECT o.customer_id,
             COUNT(DISTINCT o.order_id)                           AS total_pedidos,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total_vendas
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY o.customer_id
)
  SELECT *
    FROM vendas_cliente
   WHERE total_pedidos > 10
ORDER BY total_vendas DESC;
```

In [ ]:
sql = '''
WITH vendas_cliente AS
(
      SELECT o.customer_id,
             COUNT(DISTINCT o.order_id)                           AS total_pedidos,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total_vendas
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY o.customer_id
)
  SELECT *
    FROM vendas_cliente
   WHERE total_pedidos > 10
ORDER BY total_vendas DESC
'''

# A CTE vira simplesmente uma variável.
vendas_cliente = (pedidos.merge(itens_valorizados, on = "order_id").groupby("customer_id", as_index = False)
                  .agg(total_pedidos = ("order_id", "nunique"), total_vendas = ("valor", "sum")))

# O HAVING vira um filtro comum, aplicado depois do agrupamento.
fieis = (vendas_cliente[vendas_cliente["total_pedidos"] > 10].sort_values("total_vendas", ascending = False))

nw.conferir(sql, fieis)
fieis

Três equivalências neste exemplo:

| SQL | pandas |
|---|---|
| `WITH vendas_cliente AS (...)` | `vendas_cliente = ...` |
| `COUNT(DISTINCT o.order_id)` | `("order_id", "nunique")` |
| `HAVING` / `WHERE` sobre a CTE | filtro comum, **depois** do `groupby` |

A diferença prática entre `WHERE` e `HAVING` desaparece em pandas: como o agrupamento produz um DataFrame novo, filtrar antes ou depois é só uma questão de em qual variável você aplica o filtro.

---

## Parte 4 - Subconsultas e anti-junções

### Exemplo 9 - Subconsulta correlacionada

Lista básica 02, exercício 40 - produtos acima do preço médio da própria categoria:

```sql
SELECT p.product_name, 
       c.category_name, 
       p.unit_price
  FROM products   p
  JOIN categories c ON c.category_id = p.category_id
 WHERE p.unit_price > (SELECT AVG(p2.unit_price)
                         FROM products p2
                        WHERE p2.category_id = p.category_id)
```

In [ ]:
acima_da_media = (produtos.assign(media_categoria=produtos.groupby("category_id")["unit_price"].transform("mean"))
                  .query("unit_price > media_categoria")
                  .merge(categorias[["category_id", "category_name"]], on="category_id")
                  [["product_name", "category_name", "unit_price", "media_categoria"]]
                  .sort_values(["category_name", "unit_price"], ascending=[True, False]))

acima_da_media.head(10)

**`transform` é a peça que faltava.** Ele agrega por grupo mas devolve um valor **por linha**, alinhado ao índice original — que é exatamente o que uma subconsulta correlacionada, ou uma função de janela `AVG(...) OVER (PARTITION BY ...)`, faz.

Compare com `agg`, que devolve **uma linha por grupo**. É a distinção que mais confunde quem chega do SQL, e você tem a vantagem de já ter o modelo mental certo: `agg` é `GROUP BY`, `transform` é `OVER (PARTITION BY ...)`.

### Exemplo 10 - `NOT IN` e `NOT EXISTS`

Lista básica 02, exercício 37 - quem não é chefe de ninguém:

In [ ]:
ids_de_chefes = funcionarios["reports_to"].dropna().unique()
nao_sao_chefes = funcionarios[~funcionarios["employee_id"].isin(ids_de_chefes)]

print(f"{len(nao_sao_chefes)} funcionários não chefiam ninguém")
nao_sao_chefes[["employee_id", "first_name", "last_name"]]

Para o caso mais geral, `NOT EXISTS` sobre uma junção, o padrão é a **anti-junção** com `indicator = True`. Clientes que nunca fizeram pedido:

In [ ]:
sem_pedido = (clientes.merge(pedidos[["customer_id"]].drop_duplicates(), on = "customer_id", how = "left", indicator = True)
              .query("_merge == 'left_only'").drop(columns="_merge"))

print(f"{len(sem_pedido)} clientes sem nenhum pedido")
sem_pedido[["customer_id", "company_name", "country"]]

`indicator = True` cria a coluna `_merge` com os valores `both`, `left_only` e `right_only`. É a forma mais legível de expressar `NOT EXISTS`, `EXISTS` e junções externas completas em pandas, e o mais próximo que existe de um `FULL OUTER JOIN` com diagnóstico.

---

## Parte 5 - Funções de janela

### Exemplo 11 - `RANK() OVER (PARTITION BY ...)`

Os três produtos mais caros de cada categoria:

```sql
WITH ranqueados AS
(
    SELECT product_name, 
           category_id, 
           unit_price,
           RANK() OVER (PARTITION BY category_id ORDER BY unit_price DESC) AS posicao
      FROM products
)
SELECT * 
  FROM ranqueados 
 WHERE posicao <= 3
```

In [ ]:
ranqueados = produtos.assign(posicao = produtos.groupby("category_id")["unit_price"].rank(method = "min", ascending = False)
)

top3 = (ranqueados[ranqueados["posicao"] <= 3].merge(categorias[["category_id", "category_name"]], on = "category_id")
        .sort_values(["category_name", "posicao"])[["category_name", "product_name", "unit_price", "posicao"]])

top3.head(12)

O parâmetro `method` escolhe qual função de janela do SQL você está reproduzindo:

| SQL | pandas |
|---|---|
| `RANK()` | `rank(method = "min")` |
| `DENSE_RANK()` | `rank(method = "dense")` |
| `ROW_NUMBER()` | `rank(method = "first")` |

### Exemplo 12 - `LAG` e `DATE_TRUNC`: Variação mensal

Padrão que aparece nas suas listas avançadas: faturamento por mês e a variação contra o mês anterior.

In [ ]:
vendas_mensais = (pedidos.merge(itens_valorizados, on = "order_id").assign(mes = lambda d: d["order_date"].dt.to_period("M"))
                  .groupby("mes", as_index = False).agg(faturamento=("valor", "sum")).sort_values("mes"))

vendas_mensais["mes_anterior"] = vendas_mensais["faturamento"].shift(1)
vendas_mensais["variacao_pct"] = (vendas_mensais["faturamento"] / vendas_mensais["mes_anterior"] - 1) * 100

vendas_mensais.head(12)

`shift(1)` é o `LAG`, `shift(-1)` é o `LEAD`.

**Ordene antes de usar**: Ao contrário do SQL, onde o `ORDER BY` faz parte da cláusula `OVER`, aqui a ordem das linhas é o que define o deslocamento. Esquecer o `sort_values` é o erro mais comum, e ele não gera nenhum aviso.

Quando o deslocamento precisa respeitar um grupo (i.e. o `PARTITION BY` do `LAG`), encadeie o `groupby`: `df.groupby("g")["x"].shift(1)`.

### Exemplo 13 - `CASE WHEN` e `COALESCE`

In [ ]:
faixas = np.select([produtos["unit_price"] < 20, produtos["unit_price"] < 50], ["barato", "médio"], default = "caro",)

classificados = produtos.assign(faixa = faixas, fornecedor = produtos["supplier_id"].fillna(-1).astype("int64"),)

classificados["faixa"].value_counts()

`np.select(condicoes, valores, default=...)` é o `CASE WHEN`. As condições são avaliadas em ordem e a primeira verdadeira vence, exatamente como no SQL. Para um mapeamento simples de um para um, `.map({...})` é mais legível.

`fillna` é o `COALESCE`. Repare que `astype` é usado depois. Em pandas, uma coluna inteira com nulos vira `float64`, porque `int64` não representa `NaN`. É uma diferença real em relação ao PostgreSQL e vale ter em mente ao gravar dados de volta no banco.

---

# Exercícios

A partir daqui é com você. As consultas abaixo saíram das suas próprias listas — o gabarito está
em [`estudos/sql/respostas/`](../../sql/respostas/).

Use `nw.conferir(sql, resultado)` em cada uma para validar contra o banco.

**Meta da fase: 50 exercícios resolvidos**, além dos 13 exemplos.

### Exercício 01

Produtos descontinuados, ordenados por nome.

```sql
  SELECT product_id, 
         product_name
    FROM products
   WHERE discontinued = 1
ORDER BY product_name
```

In [ ]:
sql = '''
  SELECT product_id, 
         product_name
    FROM products
   WHERE discontinued = 1
ORDER BY product_name
'''

resultado = produtos.loc[produtos["discontinued"] == 1, ["product_id", "product_name"]].sort_values("product_name", ascending = False)

nw.conferir(sql, resultado)

### Exercício 02

Quantidade de produtos por categoria, da maior para a menor.

```sql
  SELECT c.category_name, 
         COUNT(*) AS total
    FROM products   p
    JOIN categories c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY total DESC
```

In [ ]:
sql = '''
  SELECT c.category_name, 
         COUNT(*) AS total
    FROM products   p
    JOIN categories c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY total DESC
'''

resultado = (produtos.merge(categorias, on = "category_id", how = "inner").groupby("category_name").size().reset_index(name = "total")
             .sort_values("total", ascending = False))

nw.conferir(sql, resultado)

### Exercício 03

Clientes do Brasil e da Argentina.

```sql
SELECT customer_id, company_name, country
  FROM customers
 WHERE country IN ('Brazil', 'Argentina')
```

In [ ]:
sql = '''
SELECT customer_id, company_name, country
  FROM customers
 WHERE country IN ('Brazil', 'Argentina')
'''

resultado = clientes[clientes["country"].isin(["Brazil", "Argentina"])][["customer_id", "company_name", "country"]]


nw.conferir(sql, resultado)

### Exercício 04

Pedidos ainda não enviados.

```sql
SELECT order_id, customer_id, order_date
  FROM orders
 WHERE shipped_date IS NULL
```

In [ ]:
sql = '''
SELECT order_id, customer_id, order_date
  FROM orders
 WHERE shipped_date IS NULL
'''

resultado = pedidos[pedidos["shipped_date"].isna()][["order_id", "customer_id", "order_date"]]

nw.conferir(sql, resultado)

### Exercício 05

Ticket médio por funcionário. Requer juntar três tabelas e agregar.

```sql
  SELECT e.employee_id, e.first_name, e.last_name,
         AVG(od.quantity * od.unit_price * (1 - od.discount)) AS ticket_medio
    FROM employees     e
    JOIN orders        o  ON o.employee_id = e.employee_id
    JOIN order_details od ON od.order_id   = o.order_id
GROUP BY e.employee_id, e.first_name, e.last_name
ORDER BY ticket_medio DESC
```

In [ ]:
sql = '''
  SELECT e.employee_id, e.first_name, e.last_name,
         AVG(od.quantity * od.unit_price * (1 - od.discount)) AS ticket_medio
    FROM employees     e
    JOIN orders        o  ON o.employee_id = e.employee_id
    JOIN order_details od ON od.order_id   = o.order_id
GROUP BY e.employee_id, e.first_name, e.last_name
ORDER BY ticket_medio DESC
'''

resultado = (funcionarios.merge(pedidos, on = "employee_id", how = "inner").merge(itens, on = "order_id", how = "inner")
    .assign(valor_liquido = lambda df: df["quantity"] * df["unit_price"] * (1 - df["discount"]))
    .groupby(["employee_id", "first_name", "last_name"])["valor_liquido"].mean()
    .reset_index(name = "ticket_medio")
    .sort_values("ticket_medio", ascending = False)
)

nw.conferir(sql, resultado)

### Exercício 06

Os 5 pedidos de maior valor. Lista avançada 05, exercício 02.

```sql
WITH total_pedido AS
(
      SELECT order_id, SUM(unit_price * quantity) AS total
        FROM order_details
    GROUP BY order_id
)
  SELECT *
    FROM total_pedido
ORDER BY total DESC
   LIMIT 5
```

In [ ]:
sql = '''
WITH total_pedido AS
(
      SELECT order_id, SUM(unit_price * quantity) AS total
        FROM order_details
    GROUP BY order_id
)
  SELECT *
    FROM total_pedido
ORDER BY total DESC
   LIMIT 5
'''

totalPedido = (itens.assign(subtotal = lambda df: df["unit_price"] * df["quantity"]).groupby("order_id")["subtotal"].sum()
               .reset_index(name = "total"))

resultado = (totalPedido.sort_values("total", ascending = False).head(5))

nw.conferir(sql, resultado)

### Exercício 07

Fornecedores com produtos em estoque baixo (menos de 10 unidades).

```sql
  SELECT s.company_name, COUNT(*) AS produtos_em_falta
    FROM suppliers s
    JOIN products  p ON p.supplier_id = s.supplier_id
   WHERE p.units_in_stock < 10
GROUP BY s.company_name
ORDER BY produtos_em_falta DESC
```

In [ ]:
sql = '''
  SELECT s.company_name, COUNT(*) AS produtos_em_falta
    FROM suppliers s
    JOIN products  p ON p.supplier_id = s.supplier_id
   WHERE p.units_in_stock < 10
GROUP BY s.company_name
ORDER BY produtos_em_falta DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 08

Frete médio por país de destino, apenas países acima da média global. Exige `transform`.

```sql
WITH frete_pais AS
(
      SELECT ship_country, AVG(freight) AS frete_medio
        FROM orders
    GROUP BY ship_country
)
SELECT *
  FROM frete_pais
 WHERE frete_medio > (SELECT AVG(freight) FROM orders)
```

In [ ]:
sql = '''
WITH frete_pais AS
(
      SELECT ship_country, AVG(freight) AS frete_medio
        FROM orders
    GROUP BY ship_country
)
SELECT *
  FROM frete_pais
 WHERE frete_medio > (SELECT AVG(freight) FROM orders)
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 09

Produtos que nunca foram vendidos. Anti-junção.

```sql
SELECT p.product_id, p.product_name
  FROM products p
 WHERE NOT EXISTS (SELECT 1
                     FROM order_details od
                    WHERE od.product_id = p.product_id)
```

In [ ]:
sql = '''
SELECT p.product_id, p.product_name
  FROM products p
 WHERE NOT EXISTS (SELECT 1
                     FROM order_details od
                    WHERE od.product_id = p.product_id)
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 10

Tempo médio de entrega em dias, por país. Cuidado com pedidos não enviados.

```sql
  SELECT ship_country,
         AVG(shipped_date - order_date) AS dias_medios
    FROM orders
   WHERE shipped_date IS NOT NULL
GROUP BY ship_country
ORDER BY dias_medios DESC
```

In [ ]:
sql = '''
  SELECT ship_country,
         AVG(shipped_date - order_date) AS dias_medios
    FROM orders
   WHERE shipped_date IS NOT NULL
GROUP BY ship_country
ORDER BY dias_medios DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 11

Faturamento por ano e trimestre.

```sql
  SELECT EXTRACT(YEAR    FROM o.order_date) AS ano,
         EXTRACT(QUARTER FROM o.order_date) AS trimestre,
         SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
    FROM orders        o
    JOIN order_details od ON od.order_id = o.order_id
GROUP BY 1, 2
ORDER BY 1, 2
```

In [ ]:
sql = '''
  SELECT EXTRACT(YEAR    FROM o.order_date) AS ano,
         EXTRACT(QUARTER FROM o.order_date) AS trimestre,
         SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
    FROM orders        o
    JOIN order_details od ON od.order_id = o.order_id
GROUP BY 1, 2
ORDER BY 1, 2
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 12

Participação de cada produto no faturamento da sua categoria, em porcentagem. Exige `transform` sobre o resultado de um `groupby`.

```sql
WITH vendas AS
(
      SELECT p.category_id, p.product_name,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total
        FROM order_details od
        JOIN products      p ON p.product_id = od.product_id
    GROUP BY p.category_id, p.product_name
)
SELECT category_id, product_name, total,
       100.0 * total / SUM(total) OVER (PARTITION BY category_id) AS pct_categoria
  FROM vendas
ORDER BY category_id, pct_categoria DESC
```

In [ ]:
sql = '''
WITH vendas AS
(
      SELECT p.category_id, p.product_name,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total
        FROM order_details od
        JOIN products      p ON p.product_id = od.product_id
    GROUP BY p.category_id, p.product_name
)
SELECT category_id, product_name, total,
       100.0 * total / SUM(total) OVER (PARTITION BY category_id) AS pct_categoria
  FROM vendas
ORDER BY category_id, pct_categoria DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 13

Intervalo médio, em dias, entre pedidos consecutivos de cada cliente. Exige `shift` dentro de `groupby`.

```sql
WITH ordenados AS
(
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS anterior
      FROM orders
)
  SELECT customer_id, AVG(order_date - anterior) AS dias_entre_pedidos
    FROM ordenados
   WHERE anterior IS NOT NULL
GROUP BY customer_id
ORDER BY dias_entre_pedidos
```

In [ ]:
sql = '''
WITH ordenados AS
(
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS anterior
      FROM orders
)
  SELECT customer_id, AVG(order_date - anterior) AS dias_entre_pedidos
    FROM ordenados
   WHERE anterior IS NOT NULL
GROUP BY customer_id
ORDER BY dias_entre_pedidos
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 14

Clientes que compraram em todas as categorias de produto. Compare a contagem de categorias distintas por cliente com o total de categorias.

```sql
SELECT c.customer_id, c.company_name
  FROM customers c
  JOIN orders        o  ON o.customer_id = c.customer_id
  JOIN order_details od ON od.order_id   = o.order_id
  JOIN products      p  ON p.product_id  = od.product_id
GROUP BY c.customer_id, c.company_name
HAVING COUNT(DISTINCT p.category_id) = (SELECT COUNT(*) FROM categories)
```

In [ ]:
sql = '''
SELECT c.customer_id, c.company_name
  FROM customers c
  JOIN orders        o  ON o.customer_id = c.customer_id
  JOIN order_details od ON od.order_id   = o.order_id
  JOIN products      p  ON p.product_id  = od.product_id
GROUP BY c.customer_id, c.company_name
HAVING COUNT(DISTINCT p.category_id) = (SELECT COUNT(*) FROM categories)
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 15

Hierarquia completa de funcionários com o nível de cada um. Lista avançada 05, exercício 22 — a CTE recursiva. Em pandas, resolve-se com um laço que junta o resultado a si mesmo até não sobrar ninguém.

```sql
WITH RECURSIVE hierarquia AS
(
    SELECT employee_id, first_name, last_name, reports_to, 1 AS nivel
      FROM employees
     WHERE reports_to IS NULL

     UNION ALL

    SELECT e.employee_id, e.first_name, e.last_name, e.reports_to, h.nivel + 1
      FROM employees  e
      JOIN hierarquia h ON h.employee_id = e.reports_to
)
SELECT * FROM hierarquia ORDER BY nivel, last_name
```

In [ ]:
sql = '''
WITH RECURSIVE hierarquia AS
(
    SELECT employee_id, first_name, last_name, reports_to, 1 AS nivel
      FROM employees
     WHERE reports_to IS NULL

     UNION ALL

    SELECT e.employee_id, e.first_name, e.last_name, e.reports_to, h.nivel + 1
      FROM employees  e
      JOIN hierarquia h ON h.employee_id = e.reports_to
)
SELECT * FROM hierarquia ORDER BY nivel, last_name
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 16

Escolha livre: pegue qualquer consulta da lista avançada 05 que você ainda não traduziu e faça a ponte.

```sql
-- sua escolha
```

In [ ]:
sql = '''
-- sua escolha
'''

# resultado = ...

# nw.conferir(sql, resultado)

---

# Exercícios 17 a 50

A segunda leva introduz partes do pandas que os exemplos não cobriram: texto com expressão regular,
tabelas cruzadas, `melt`, janelas móveis, `qcut`, `cut` e aritmética de datas. Por isso a maioria
traz uma **dica** apontando a ferramenta — mas não a solução.

Em alguns exercícios o SQL e o pandas **divergem de propósito** nos detalhes — `NTILE` contra
`qcut`, `EXTRACT(DOW)` contra `dayofweek`. Nesses casos o `nw.conferir` vai mostrar o mesmo número
de linhas com valores diferentes. Entender por quê faz parte do exercício.

## 17–22 · Texto, nulos e condições compostas

### Exercício 17

Clientes cujo contato tem o cargo `Owner`, ordenados por país e nome.

```sql
  SELECT customer_id, company_name, contact_name, country
    FROM customers
   WHERE contact_title = 'Owner'
ORDER BY country, company_name
```

In [ ]:
sql = '''
  SELECT customer_id, company_name, contact_name, country
    FROM customers
   WHERE contact_title = 'Owner'
ORDER BY country, company_name
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 18

Produtos com `chef` no nome, sem diferenciar maiúsculas de minúsculas. Dica: `str.contains(..., case=False)`.

```sql
SELECT product_id, product_name
  FROM products
 WHERE product_name ILIKE '%chef%'
```

In [ ]:
sql = '''
SELECT product_id, product_name
  FROM products
 WHERE product_name ILIKE '%chef%'
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 19

Idade de cada funcionário na data de contratação, em anos completos. Dica: dividir a diferença em dias por 365 erra perto do aniversário — compare ano, mês e dia.

```sql
  SELECT employee_id, first_name, last_name,
         EXTRACT(YEAR FROM AGE(hire_date, birth_date))::int AS idade_na_contratacao
    FROM employees
ORDER BY idade_na_contratacao
```

In [ ]:
sql = '''
  SELECT employee_id, first_name, last_name,
         EXTRACT(YEAR FROM AGE(hire_date, birth_date))::int AS idade_na_contratacao
    FROM employees
ORDER BY idade_na_contratacao
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 20

Clientes sem região **e** sem fax. Dica: combine as máscaras com `&`, cada uma entre parênteses.

```sql
SELECT customer_id, company_name, country
  FROM customers
 WHERE region IS NULL
   AND fax    IS NULL
```

In [ ]:
sql = '''
SELECT customer_id, company_name, country
  FROM customers
 WHERE region IS NULL
   AND fax    IS NULL
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 21

Telefone dos fornecedores mantendo apenas os dígitos. Dica: `str.replace(..., regex=True)`.

```sql
SELECT supplier_id, company_name, phone,
       REGEXP_REPLACE(phone, '[^0-9]', '', 'g') AS telefone_digitos
  FROM suppliers
```

In [ ]:
sql = '''
SELECT supplier_id, company_name, phone,
       REGEXP_REPLACE(phone, '[^0-9]', '', 'g') AS telefone_digitos
  FROM suppliers
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 22

Produtos ativos que precisam de reposição: estoque somado ao que já está encomendado não passa do nível de reposição.

```sql
  SELECT product_id, product_name, units_in_stock, units_on_order, reorder_level
    FROM products
   WHERE discontinued = 0
     AND units_in_stock + units_on_order <= reorder_level
ORDER BY units_in_stock
```

In [ ]:
sql = '''
  SELECT product_id, product_name, units_in_stock, units_on_order, reorder_level
    FROM products
   WHERE discontinued = 0
     AND units_in_stock + units_on_order <= reorder_level
ORDER BY units_in_stock
'''

# resultado = ...

# nw.conferir(sql, resultado)

## 23–30 · Agregação

### Exercício 23

Países com mais de 5 clientes.

```sql
  SELECT country, COUNT(*) AS clientes
    FROM customers
GROUP BY country
  HAVING COUNT(*) > 5
ORDER BY clientes DESC
```

In [ ]:
sql = '''
  SELECT country, COUNT(*) AS clientes
    FROM customers
GROUP BY country
  HAVING COUNT(*) > 5
ORDER BY clientes DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 24

Preço mínimo, máximo, médio e desvio-padrão por categoria. Dica: várias agregações nomeadas no mesmo `agg`. O `std` do pandas é amostral, igual ao `STDDEV` do PostgreSQL.

```sql
  SELECT c.category_name,
         MIN(p.unit_price)    AS minimo,
         MAX(p.unit_price)    AS maximo,
         AVG(p.unit_price)    AS media,
         STDDEV(p.unit_price) AS desvio_padrao
    FROM products   p
    JOIN categories c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY media DESC
```

In [ ]:
sql = '''
  SELECT c.category_name,
         MIN(p.unit_price)    AS minimo,
         MAX(p.unit_price)    AS maximo,
         AVG(p.unit_price)    AS media,
         STDDEV(p.unit_price) AS desvio_padrao
    FROM products   p
    JOIN categories c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY media DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 25

Os 10 fornecedores com maior valor parado em estoque.

```sql
  SELECT s.company_name,
         SUM(p.unit_price * p.units_in_stock) AS valor_estoque
    FROM suppliers s
    JOIN products  p ON p.supplier_id = s.supplier_id
GROUP BY s.company_name
ORDER BY valor_estoque DESC
   LIMIT 10
```

In [ ]:
sql = '''
  SELECT s.company_name,
         SUM(p.unit_price * p.units_in_stock) AS valor_estoque
    FROM suppliers s
    JOIN products  p ON p.supplier_id = s.supplier_id
GROUP BY s.company_name
ORDER BY valor_estoque DESC
   LIMIT 10
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 26

Pedidos, frete total e frete médio por transportadora. Dica: a chave tem nomes diferentes nas duas tabelas — `left_on` e `right_on`.

```sql
  SELECT sh.company_name,
         COUNT(*)       AS pedidos,
         SUM(o.freight) AS frete_total,
         AVG(o.freight) AS frete_medio
    FROM orders   o
    JOIN shippers sh ON sh.shipper_id = o.ship_via
GROUP BY sh.company_name
ORDER BY frete_total DESC
```

In [ ]:
sql = '''
  SELECT sh.company_name,
         COUNT(*)       AS pedidos,
         SUM(o.freight) AS frete_total,
         AVG(o.freight) AS frete_medio
    FROM orders   o
    JOIN shippers sh ON sh.shipper_id = o.ship_via
GROUP BY sh.company_name
ORDER BY frete_total DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 27

Média e mediana de unidades por pedido. Dica: agregue em dois níveis — primeiro por pedido, depois sobre o resultado.

```sql
WITH por_pedido AS
(
      SELECT order_id, SUM(quantity) AS unidades
        FROM order_details
    GROUP BY order_id
)
SELECT AVG(unidades)                                         AS media,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY unidades) AS mediana
  FROM por_pedido
```

In [ ]:
sql = '''
WITH por_pedido AS
(
      SELECT order_id, SUM(quantity) AS unidades
        FROM order_details
    GROUP BY order_id
)
SELECT AVG(unidades)                                         AS media,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY unidades) AS mediana
  FROM por_pedido
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 28

Percentual de pedidos entregues com atraso, por país de destino. Dica: a média de uma coluna booleana é uma proporção.

```sql
  SELECT ship_country,
         COUNT(*) AS pedidos_enviados,
         100.0 * AVG(CASE WHEN shipped_date > required_date THEN 1 ELSE 0 END) AS pct_atraso
    FROM orders
   WHERE shipped_date IS NOT NULL
GROUP BY ship_country
ORDER BY pct_atraso DESC
```

In [ ]:
sql = '''
  SELECT ship_country,
         COUNT(*) AS pedidos_enviados,
         100.0 * AVG(CASE WHEN shipped_date > required_date THEN 1 ELSE 0 END) AS pct_atraso
    FROM orders
   WHERE shipped_date IS NOT NULL
GROUP BY ship_country
ORDER BY pct_atraso DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 29

Faturamento por país do cliente e a participação de cada país no total.

```sql
WITH por_pais AS
(
      SELECT c.country,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM customers     c
        JOIN orders        o  ON o.customer_id = c.customer_id
        JOIN order_details od ON od.order_id   = o.order_id
    GROUP BY c.country
)
  SELECT country, faturamento,
         100.0 * faturamento / SUM(faturamento) OVER () AS pct_total
    FROM por_pais
ORDER BY faturamento DESC
```

In [ ]:
sql = '''
WITH por_pais AS
(
      SELECT c.country,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM customers     c
        JOIN orders        o  ON o.customer_id = c.customer_id
        JOIN order_details od ON od.order_id   = o.order_id
    GROUP BY c.country
)
  SELECT country, faturamento,
         100.0 * faturamento / SUM(faturamento) OVER () AS pct_total
    FROM por_pais
ORDER BY faturamento DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 30

Pedidos de cada funcionário por ano, com um ano em cada coluna. Dica: `pd.crosstab` ou `pivot_table`.

```sql
  SELECT e.first_name || ' ' || e.last_name AS funcionario,
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1996) AS "1996",
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1997) AS "1997",
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1998) AS "1998"
    FROM employees e
    JOIN orders    o ON o.employee_id = e.employee_id
GROUP BY funcionario
ORDER BY funcionario
```

In [ ]:
sql = '''
  SELECT e.first_name || ' ' || e.last_name AS funcionario,
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1996) AS "1996",
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1997) AS "1997",
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1998) AS "1998"
    FROM employees e
    JOIN orders    o ON o.employee_id = e.employee_id
GROUP BY funcionario
ORDER BY funcionario
'''

# resultado = ...

# nw.conferir(sql, resultado)

## 31–36 · Junções e operações de conjunto

### Exercício 31

Territórios atendidos por cada funcionário, com a região. Dica: é uma relação muitos-para-muitos — duas junções passando pela tabela associativa `employee_territories`.

```sql
  SELECT e.first_name, e.last_name,
         t.territory_description AS territorio,
         r.region_description    AS regiao
    FROM employees            e
    JOIN employee_territories et ON et.employee_id = e.employee_id
    JOIN territories          t  ON t.territory_id = et.territory_id
    JOIN region               r  ON r.region_id    = t.region_id
ORDER BY e.last_name, territorio
```

In [ ]:
sql = '''
  SELECT e.first_name, e.last_name,
         t.territory_description AS territorio,
         r.region_description    AS regiao
    FROM employees            e
    JOIN employee_territories et ON et.employee_id = e.employee_id
    JOIN territories          t  ON t.territory_id = et.territory_id
    JOIN region               r  ON r.region_id    = t.region_id
ORDER BY e.last_name, territorio
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 32

Clientes e fornecedores localizados na mesma cidade e país. Dica: `merge` com duas chaves, `on=["city", "country"]`.

```sql
  SELECT c.city, c.company_name AS cliente, s.company_name AS fornecedor
    FROM customers c
    JOIN suppliers s ON s.city = c.city AND s.country = c.country
ORDER BY c.city, cliente
```

In [ ]:
sql = '''
  SELECT c.city, c.company_name AS cliente, s.company_name AS fornecedor
    FROM customers c
    JOIN suppliers s ON s.city = c.city AND s.country = c.country
ORDER BY c.city, cliente
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 33

Produtos ativos e descontinuados por categoria, lado a lado. Dica: agregação condicional — some uma coluna booleana, ou use `pd.crosstab`.

```sql
  SELECT c.category_name,
         COUNT(*) FILTER (WHERE p.discontinued = 0) AS ativos,
         COUNT(*) FILTER (WHERE p.discontinued = 1) AS descontinuados
    FROM categories c
    JOIN products   p ON p.category_id = c.category_id
GROUP BY c.category_name
ORDER BY c.category_name
```

In [ ]:
sql = '''
  SELECT c.category_name,
         COUNT(*) FILTER (WHERE p.discontinued = 0) AS ativos,
         COUNT(*) FILTER (WHERE p.discontinued = 1) AS descontinuados
    FROM categories c
    JOIN products   p ON p.category_id = c.category_id
GROUP BY c.category_name
ORDER BY c.category_name
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 34

Lista única de contatos de clientes e fornecedores, com uma coluna indicando o tipo. Dica: `pd.concat` exige as mesmas colunas nos dois lados.

```sql
  SELECT 'cliente' AS tipo, company_name, contact_name, country FROM customers
   UNION ALL
  SELECT 'fornecedor', company_name, contact_name, country FROM suppliers
ORDER BY country, company_name
```

In [ ]:
sql = '''
  SELECT 'cliente' AS tipo, company_name, contact_name, country FROM customers
   UNION ALL
  SELECT 'fornecedor', company_name, contact_name, country FROM suppliers
ORDER BY country, company_name
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 35

Pedidos entregues em uma cidade diferente da cidade cadastrada do cliente.

```sql
SELECT o.order_id, c.company_name, c.city AS cidade_cliente, o.ship_city
  FROM orders    o
  JOIN customers c ON c.customer_id = o.customer_id
 WHERE o.ship_city <> c.city
```

In [ ]:
sql = '''
SELECT o.order_id, c.company_name, c.city AS cidade_cliente, o.ship_city
  FROM orders    o
  JOIN customers c ON c.customer_id = o.customer_id
 WHERE o.ship_city <> c.city
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 36

Países que têm clientes mas nenhum fornecedor, e vice-versa. Dica: `merge(how="outer", indicator=True)`.

```sql
WITH pc AS (SELECT DISTINCT country FROM customers),
     pf AS (SELECT DISTINCT country FROM suppliers)
  SELECT COALESCE(pc.country, pf.country) AS pais,
         CASE WHEN pf.country IS NULL THEN 'só clientes'
              ELSE 'só fornecedores' END AS situacao
    FROM pc
    FULL OUTER JOIN pf ON pf.country = pc.country
   WHERE pc.country IS NULL OR pf.country IS NULL
ORDER BY situacao, pais
```

In [ ]:
sql = '''
WITH pc AS (SELECT DISTINCT country FROM customers),
     pf AS (SELECT DISTINCT country FROM suppliers)
  SELECT COALESCE(pc.country, pf.country) AS pais,
         CASE WHEN pf.country IS NULL THEN 'só clientes'
              ELSE 'só fornecedores' END AS situacao
    FROM pc
    FULL OUTER JOIN pf ON pf.country = pc.country
   WHERE pc.country IS NULL OR pf.country IS NULL
ORDER BY situacao, pais
'''

# resultado = ...

# nw.conferir(sql, resultado)

## 37–44 · Funções de janela

### Exercício 37

Faturamento mensal e o acumulado dentro de cada ano. Dica: `groupby(ano).cumsum()` sobre a série já ordenada.

```sql
WITH mensal AS
(
      SELECT DATE_TRUNC('month', o.order_date)::date AS mes,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY 1
)
  SELECT mes, faturamento,
         SUM(faturamento) OVER (PARTITION BY EXTRACT(YEAR FROM mes) ORDER BY mes) AS acumulado_ano
    FROM mensal
ORDER BY mes
```

In [ ]:
sql = '''
WITH mensal AS
(
      SELECT DATE_TRUNC('month', o.order_date)::date AS mes,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY 1
)
  SELECT mes, faturamento,
         SUM(faturamento) OVER (PARTITION BY EXTRACT(YEAR FROM mes) ORDER BY mes) AS acumulado_ano
    FROM mensal
ORDER BY mes
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 38

Média móvel de 3 meses do faturamento. Dica: `rolling(3)`. O SQL calcula a média mesmo nos dois primeiros meses, com a janela incompleta — em pandas isso é `min_periods=1`.

```sql
WITH mensal AS
(
      SELECT DATE_TRUNC('month', o.order_date)::date AS mes,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY 1
)
  SELECT mes, faturamento,
         AVG(faturamento) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS media_movel_3m
    FROM mensal
ORDER BY mes
```

In [ ]:
sql = '''
WITH mensal AS
(
      SELECT DATE_TRUNC('month', o.order_date)::date AS mes,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY 1
)
  SELECT mes, faturamento,
         AVG(faturamento) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS media_movel_3m
    FROM mensal
ORDER BY mes
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 39

Primeiro pedido de cada cliente. Dica: `ROW_NUMBER() = 1` é ordenar e depois `drop_duplicates(subset=...)` ou `groupby(...).head(1)`.

```sql
WITH numerados AS
(
    SELECT customer_id, order_id, order_date,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS n
      FROM orders
)
  SELECT customer_id, order_id, order_date
    FROM numerados
   WHERE n = 1
ORDER BY order_date, customer_id
```

In [ ]:
sql = '''
WITH numerados AS
(
    SELECT customer_id, order_id, order_date,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS n
      FROM orders
)
  SELECT customer_id, order_id, order_date
    FROM numerados
   WHERE n = 1
ORDER BY order_date, customer_id
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 40

Ranking dos funcionários por faturamento dentro de cada ano. Dica: `rank(method="dense")` dentro do `groupby`.

```sql
WITH por_ano AS
(
      SELECT EXTRACT(YEAR FROM o.order_date)::int AS ano,
             e.first_name || ' ' || e.last_name  AS funcionario,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN employees     e  ON e.employee_id = o.employee_id
        JOIN order_details od ON od.order_id   = o.order_id
    GROUP BY 1, 2
)
  SELECT ano, funcionario, faturamento,
         DENSE_RANK() OVER (PARTITION BY ano ORDER BY faturamento DESC) AS posicao
    FROM por_ano
ORDER BY ano, posicao
```

In [ ]:
sql = '''
WITH por_ano AS
(
      SELECT EXTRACT(YEAR FROM o.order_date)::int AS ano,
             e.first_name || ' ' || e.last_name  AS funcionario,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN employees     e  ON e.employee_id = o.employee_id
        JOIN order_details od ON od.order_id   = o.order_id
    GROUP BY 1, 2
)
  SELECT ano, funcionario, faturamento,
         DENSE_RANK() OVER (PARTITION BY ano ORDER BY faturamento DESC) AS posicao
    FROM por_ano
ORDER BY ano, posicao
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 41

Frete de cada pedido comparado à média do seu país de destino. Dica: `transform("mean")` — a tabela não pode perder linhas.

```sql
  SELECT order_id, ship_country, freight,
         AVG(freight) OVER (PARTITION BY ship_country)           AS media_pais,
         freight - AVG(freight) OVER (PARTITION BY ship_country) AS diferenca
    FROM orders
ORDER BY diferenca DESC
```

In [ ]:
sql = '''
  SELECT order_id, ship_country, freight,
         AVG(freight) OVER (PARTITION BY ship_country)           AS media_pais,
         freight - AVG(freight) OVER (PARTITION BY ship_country) AS diferenca
    FROM orders
ORDER BY diferenca DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 42

Divida os clientes em quartis de faturamento, quartil 1 sendo o de maior valor. Dica: `pd.qcut(..., 4, labels=[...])`, com os rótulos invertidos. Compare as contagens por quartil com as do `NTILE`: os dois repartem as sobras de forma diferente.

```sql
WITH por_cliente AS
(
      SELECT o.customer_id,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY o.customer_id
)
  SELECT customer_id, faturamento,
         NTILE(4) OVER (ORDER BY faturamento DESC) AS quartil
    FROM por_cliente
ORDER BY quartil, faturamento DESC
```

In [ ]:
sql = '''
WITH por_cliente AS
(
      SELECT o.customer_id,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY o.customer_id
)
  SELECT customer_id, faturamento,
         NTILE(4) OVER (ORDER BY faturamento DESC) AS quartil
    FROM por_cliente
ORDER BY quartil, faturamento DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 43

Curva ABC dos produtos: classe A até 80% do faturamento acumulado, B até 95%, C o restante. Dica: ordene, faça `cumsum` da participação e classifique com `pd.cut` ou `np.select`.

```sql
WITH por_produto AS
(
      SELECT p.product_name,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM order_details od
        JOIN products      p ON p.product_id = od.product_id
    GROUP BY p.product_name
),
acumulado AS
(
    SELECT product_name, faturamento,
           100.0 * SUM(faturamento) OVER (ORDER BY faturamento DESC)
                 / SUM(faturamento) OVER ()                   AS pct_acumulado
      FROM por_produto
)
  SELECT product_name, faturamento, pct_acumulado,
         CASE WHEN pct_acumulado <= 80 THEN 'A'
              WHEN pct_acumulado <= 95 THEN 'B'
              ELSE 'C' END AS classe
    FROM acumulado
ORDER BY faturamento DESC
```

In [ ]:
sql = '''
WITH por_produto AS
(
      SELECT p.product_name,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM order_details od
        JOIN products      p ON p.product_id = od.product_id
    GROUP BY p.product_name
),
acumulado AS
(
    SELECT product_name, faturamento,
           100.0 * SUM(faturamento) OVER (ORDER BY faturamento DESC)
                 / SUM(faturamento) OVER ()                   AS pct_acumulado
      FROM por_produto
)
  SELECT product_name, faturamento, pct_acumulado,
         CASE WHEN pct_acumulado <= 80 THEN 'A'
              WHEN pct_acumulado <= 95 THEN 'B'
              ELSE 'C' END AS classe
    FROM acumulado
ORDER BY faturamento DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 44

Crescimento anual do faturamento de cada categoria. Dica: `groupby(...).pct_change()`. **Antes de interpretar:** 1996 começa em julho e 1998 termina em maio. O que isso faz com os percentuais?

```sql
WITH por_ano AS
(
      SELECT c.category_name,
             EXTRACT(YEAR FROM o.order_date)::int AS ano,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN order_details od ON od.order_id   = o.order_id
        JOIN products      p  ON p.product_id  = od.product_id
        JOIN categories    c  ON c.category_id = p.category_id
    GROUP BY 1, 2
)
  SELECT category_name, ano, faturamento,
         100.0 * (faturamento / LAG(faturamento) OVER (PARTITION BY category_name ORDER BY ano) - 1) AS crescimento_pct
    FROM por_ano
ORDER BY category_name, ano
```

In [ ]:
sql = '''
WITH por_ano AS
(
      SELECT c.category_name,
             EXTRACT(YEAR FROM o.order_date)::int AS ano,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
        FROM orders        o
        JOIN order_details od ON od.order_id   = o.order_id
        JOIN products      p  ON p.product_id  = od.product_id
        JOIN categories    c  ON c.category_id = p.category_id
    GROUP BY 1, 2
)
  SELECT category_name, ano, faturamento,
         100.0 * (faturamento / LAG(faturamento) OVER (PARTITION BY category_name ORDER BY ano) - 1) AS crescimento_pct
    FROM por_ano
ORDER BY category_name, ano
'''

# resultado = ...

# nw.conferir(sql, resultado)

## 45–47 · Datas

### Exercício 45

Pedidos por dia da semana. Dica: `dt.dayofweek` começa na segunda-feira = 0; o `EXTRACT(DOW)` do PostgreSQL começa no domingo = 0. Os números não vão bater sem um ajuste.

```sql
  SELECT EXTRACT(DOW FROM order_date)::int AS dia_semana,
         COUNT(*)                           AS pedidos
    FROM orders
GROUP BY 1
ORDER BY 1
```

In [ ]:
sql = '''
  SELECT EXTRACT(DOW FROM order_date)::int AS dia_semana,
         COUNT(*)                           AS pedidos
    FROM orders
GROUP BY 1
ORDER BY 1
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 46

Clientes inativos: último pedido mais de 6 meses antes do último pedido da base. Dica: em dados históricos, o "hoje" é a data mais recente do dataset, não `pd.Timestamp.now()`. Use `pd.DateOffset(months=6)`.

```sql
WITH ultimo AS
(
      SELECT customer_id, MAX(order_date) AS ultimo_pedido
        FROM orders
    GROUP BY customer_id
)
  SELECT u.customer_id, c.company_name, u.ultimo_pedido
    FROM ultimo    u
    JOIN customers c ON c.customer_id = u.customer_id
   WHERE u.ultimo_pedido < (SELECT MAX(order_date) FROM orders) - INTERVAL '6 months'
ORDER BY u.ultimo_pedido
```

In [ ]:
sql = '''
WITH ultimo AS
(
      SELECT customer_id, MAX(order_date) AS ultimo_pedido
        FROM orders
    GROUP BY customer_id
)
  SELECT u.customer_id, c.company_name, u.ultimo_pedido
    FROM ultimo    u
    JOIN customers c ON c.customer_id = u.customer_id
   WHERE u.ultimo_pedido < (SELECT MAX(order_date) FROM orders) - INTERVAL '6 months'
ORDER BY u.ultimo_pedido
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 47

Pedidos por faixa de prazo de envio: até 7 dias, 8 a 14, 15 a 30 e mais de 30. Dica: `pd.cut` com `bins` e `labels`. Atenção aos limites: `pd.cut` fecha o intervalo à direita por padrão.

```sql
WITH prazos AS
(
    SELECT shipped_date - order_date AS dias
      FROM orders
     WHERE shipped_date IS NOT NULL
)
  SELECT CASE WHEN dias <= 7  THEN '0-7'
              WHEN dias <= 14 THEN '8-14'
              WHEN dias <= 30 THEN '15-30'
              ELSE '31+' END AS faixa,
         COUNT(*) AS pedidos
    FROM prazos
GROUP BY 1
ORDER BY MIN(dias)
```

In [ ]:
sql = '''
WITH prazos AS
(
    SELECT shipped_date - order_date AS dias
      FROM orders
     WHERE shipped_date IS NOT NULL
)
  SELECT CASE WHEN dias <= 7  THEN '0-7'
              WHEN dias <= 14 THEN '8-14'
              WHEN dias <= 30 THEN '15-30'
              ELSE '31+' END AS faixa,
         COUNT(*) AS pedidos
    FROM prazos
GROUP BY 1
ORDER BY MIN(dias)
'''

# resultado = ...

# nw.conferir(sql, resultado)

## 48–50 · Reestruturação e análise

### Exercício 48

Desfaça o exercício 30: volte a tabela larga para o formato longo, uma linha por funcionário e ano. Dica: `melt`, o inverso do `pivot_table`.

```sql
WITH largo AS
(
  SELECT e.first_name || ' ' || e.last_name AS funcionario,
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1996) AS "1996",
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1997) AS "1997",
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1998) AS "1998"
    FROM employees e
    JOIN orders    o ON o.employee_id = e.employee_id
GROUP BY funcionario
)
  SELECT l.funcionario, v.ano, v.pedidos
    FROM largo l
   CROSS JOIN LATERAL (VALUES (1996, l."1996"),
                              (1997, l."1997"),
                              (1998, l."1998")) AS v(ano, pedidos)
ORDER BY l.funcionario, v.ano
```

In [ ]:
sql = '''
WITH largo AS
(
  SELECT e.first_name || ' ' || e.last_name AS funcionario,
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1996) AS "1996",
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1997) AS "1997",
         COUNT(*) FILTER (WHERE EXTRACT(YEAR FROM o.order_date) = 1998) AS "1998"
    FROM employees e
    JOIN orders    o ON o.employee_id = e.employee_id
GROUP BY funcionario
)
  SELECT l.funcionario, v.ano, v.pedidos
    FROM largo l
   CROSS JOIN LATERAL (VALUES (1996, l."1996"),
                              (1997, l."1997"),
                              (1998, l."1998")) AS v(ano, pedidos)
ORDER BY l.funcionario, v.ano
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 49

Os 10 pares de produtos que mais aparecem juntos no mesmo pedido. Dica: `merge` de `order_details` com ela mesma por `order_id`, mantendo só os pares em que o primeiro `product_id` é menor — senão cada par conta duas vezes.

```sql
  SELECT p1.product_name AS produto_a,
         p2.product_name AS produto_b,
         COUNT(*)        AS pedidos_juntos
    FROM order_details a
    JOIN order_details b  ON b.order_id = a.order_id AND b.product_id > a.product_id
    JOIN products      p1 ON p1.product_id = a.product_id
    JOIN products      p2 ON p2.product_id = b.product_id
GROUP BY p1.product_name, p2.product_name
ORDER BY pedidos_juntos DESC, produto_a, produto_b
   LIMIT 10
```

In [ ]:
sql = '''
  SELECT p1.product_name AS produto_a,
         p2.product_name AS produto_b,
         COUNT(*)        AS pedidos_juntos
    FROM order_details a
    JOIN order_details b  ON b.order_id = a.order_id AND b.product_id > a.product_id
    JOIN products      p1 ON p1.product_id = a.product_id
    JOIN products      p2 ON p2.product_id = b.product_id
GROUP BY p1.product_name, p2.product_name
ORDER BY pedidos_juntos DESC, produto_a, produto_b
   LIMIT 10
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 50

**Desafio final — segmentação RFM.** Para cada cliente: recência (dias entre o último pedido e a data mais recente da base), frequência (pedidos) e valor (faturamento), cada um pontuado de 1 a 5, com 5 sendo o melhor. Dica: combina agregação, datas e `pd.qcut`. Na recência, menos dias é melhor — inverta os rótulos. Se o `qcut` reclamar de cortes repetidos por causa de empates, aplique `rank(method="first")` antes.

```sql
WITH base AS
(
      SELECT o.customer_id,
             (SELECT MAX(order_date) FROM orders) - MAX(o.order_date) AS recencia,
             COUNT(DISTINCT o.order_id)                               AS frequencia,
             SUM(od.quantity * od.unit_price * (1 - od.discount))     AS valor
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY o.customer_id
)
  SELECT customer_id, recencia, frequencia, valor,
         NTILE(5) OVER (ORDER BY recencia DESC) AS r,
         NTILE(5) OVER (ORDER BY frequencia)    AS f,
         NTILE(5) OVER (ORDER BY valor)         AS m
    FROM base
ORDER BY r DESC, f DESC, m DESC
```

In [ ]:
sql = '''
WITH base AS
(
      SELECT o.customer_id,
             (SELECT MAX(order_date) FROM orders) - MAX(o.order_date) AS recencia,
             COUNT(DISTINCT o.order_id)                               AS frequencia,
             SUM(od.quantity * od.unit_price * (1 - od.discount))     AS valor
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY o.customer_id
)
  SELECT customer_id, recencia, frequencia, valor,
         NTILE(5) OVER (ORDER BY recencia DESC) AS r,
         NTILE(5) OVER (ORDER BY frequencia)    AS f,
         NTILE(5) OVER (ORDER BY valor)         AS m
    FROM base
ORDER BY r DESC, f DESC, m DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

---

## Fechamento da fase

Quando tiver os 50 exercícios resolvidos e conferindo, você terminou a Fase 1. O sinal de que a fase
pegou não é o número — é você conseguir olhar uma consulta e já enxergar o encadeamento de pandas
antes de escrever a primeira linha.

**Três coisas que vale registrar enquanto traduz**, porque vão virar conteúdo de entrevista:

1. **Onde o pandas ficou mais claro que o SQL** — normalmente em transformações encadeadas, onde
   cada passo fica em uma variável com nome.
2. **Onde o SQL ficou mais claro** — normalmente em junções múltiplas e em recursão.
3. **Onde os dois divergiram no resultado** — quase sempre por causa de `NULL` contra `NaN`, ou
   de tipo inteiro virando ponto flutuante. Essa é a diferença semântica de verdade entre as duas
   ferramentas, e saber explicá-la vale mais do que decorar a API.

**Próximo passo:** Fase 2 — análise exploratória e visualização, com o Projeto 1 em
[`projetos/`](../../../projetos/).